# Tutorial 8: Can two models have the same AUC but different-quality probabilities?

## Briefing for the responsible study group

| Planning item | Allocation |
|---|---|
| Presentation date | Tuesday 1 December 2026 |
| Prerequisite lecture | Model Evaluation, 20 November 2026 |
| Preparation time after prerequisite | 6 full weekdays, excluding lecture and tutorial dates |
| Format | 15-minute presentation + approximately 15-minute discussion |

**Lecture connection:** Model Evaluation slides 19–23: ROC-AUC as a ranking indicator; Classification slides 34–35: probability-based loss.

This is your group's working notebook. It replaces the requirement to augment a section of an existing course notebook. The supplied cells establish a starting point; your group develops and teaches the investigation.

### Your teaching challenge

Show what ROC-AUC tells us and what it cannot establish about probability predictions. Construct predictors that rank every case in the same order but report different probabilities, then explain why their probability quality can differ.

Use probabilities generated to be calibrated in the population. Apply a strictly increasing transformation p_new = sigmoid(a * logit(p) + b), with a > 0, to the **same cases**. This changes confidence or shifts the probability level while preserving ranking. You are comparing alternative reported predictions, not training several independent models.

### Core investigation

1. Compare three prediction versions: the original probabilities, a more extreme version (for example a=2, b=0), and a shifted version (for example a=1, b=1). State each transformation clearly.
2. On identical labels, report ROC-AUC, log-loss and the binary Brier score. Check whether rankings are preserved and explain why AUC should be unchanged apart from numerical issues.
3. Build reliability plots comparing average predicted probability with observed event frequency in bins. Include bin counts and avoid strong conclusions from sparsely populated bins. Explain population calibration versus finite-sample fluctuation.
4. Explain why a model can discriminate well while systematically overstating or understating risk. Connect your findings to uses where numerical probabilities matter, such as expected event counts or decisions based on explicit costs.

### What classmates should gain

They should distinguish discrimination from calibration and broader probability quality. Log-loss and Brier score assess more than calibration alone; do not describe either as a pure calibration metric. The main teaching contribution is the controlled counterexample to the idea that a strong AUC guarantees reliable probabilities.

### Boundaries and support

The data, transformation helper and reliability-table helper are supplied. Use the three specified prediction versions; no calibration algorithm, trainable classifier, parameter tuning or model search is needed. Tutorial 4 concerns individual losses and optimization; yours concerns ranking invariance and aggregate probability assessment. **Optional:** compare predicted and actual total events, with a discussion of sampling uncertainty.

## How to turn your investigation into a tutorial

Your group has **15 minutes to present**, followed by approximately **15 minutes of discussion** with the instructor and class. Your audience has attended the prerequisite lecture but has not done your investigation. Help them understand one important question through a focused, reproducible demonstration.

**Required contribution.** Complete the core investigation below, explain why the results occur, and discuss at least one limitation or contrasting condition. Select two or three main figures or tables that make your argument clear. A useful tutorial connects the question, experimental design, evidence and conclusion; a sequence of unexplained code cells is insufficient. You are not expected to build a production system, implement algorithms from scratch, collect external data or run an extensive model search.

**What to submit.** Submit your completed `.ipynb` notebook with explanations and reproducible code, plus any slides or other presentation material you use. Follow the instructor's announced submission deadline. Keep the supplied briefing, but edit, reorganize or replace the working sections as needed. Identify substantial changes to supplied helpers. Record package versions, seeds and relevant settings; restart the kernel and run all cells before submission. Clearly distinguish supplied material from your group's contribution.

**Presentation format is your choice.** You may present from the notebook, create slides, render to HTML, develop an interactive demonstration or combine these. Make the selected format work within 15 minutes and have a fallback for live demonstrations. Creativity earns credit when it improves understanding, engagement or the quality of the investigation. Technical spectacle should be avoided. A clear static plot can be excellent.

**AI assistance is allowed.** You may use AI to help develop code, explanations and visualizations. Your group remains responsible for correctness and must be able to explain the submitted work. Include a short account of the tools used, their main contributions, and how you checked those contributions. Verify at least one important result independently—for example, with a hand calculation, an analytic special case or a comparison with a trusted library function. A long prompt transcript is not required. Using AI is optional and is not itself rewarded.

**Assessment.** This component contributes **15% of the course grade**. Within the tutorial, the criteria are **conceptual understanding (40%)**, **added value for classmates (40%)**, and **quality of communication (20%)**. Added value comes from the investigation and how it develops understanding beyond the lecture and the supplied starter cells. Optional extensions are not required for an excellent grade. All groups should prioritize a sound core investigation over breadth.

**Suggested presentation rhythm:** about 3 minutes for the question and essential concepts, 8 minutes for the demonstration and evidence, and 4 minutes for interpretation and limitations. Prepare one question that invites the class to reason about your findings. Anticipate questions about experimental controls, assumptions and alternative explanations.

## Preliminaries: environment

The course environment provides all libraries needed to execute the starter code (see the [README for installation instructions](https://github.com/stefanlessmann/ESMT_IML)). You can install additional libraries if needed.


Run the supplied cells first. They provide a starting point, **not a completed tutorial**. Code cells marked `YOUR GROUP'S WORK` are intentionally unfinished and can be replaced or expanded.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

SEED = 888
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.2, 'font.size': 11})
print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}')

## Preliminaries: supplied data and helpers

Original probabilities are bounded away from zero and one before sampling labels. The transformation helper requires positive a to preserve score order. Small clipping is a numerical safeguard; extreme choices can induce ties, so stay with moderate parameters. Equal-width reliability bins are supplied as a starting point. Report their counts and discuss whether alternative binning changes the visual impression. Omit empty bins from the plotted points; their mean prediction and event frequency are undefined. State a convention for sparse bins, for example using hollow markers for bins with fewer than 20 cases and showing their counts. This cutoff is a display convention, not a statistical significance threshold; do not interpret noisy tail bins as decisive evidence of miscalibration.

In [ ]:
from scipy.special import expit, logit
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

rng = np.random.default_rng(SEED)
p_original = np.clip(rng.beta(1.8, 5.0, 6000), 1e-6, 1 - 1e-6)
y = rng.binomial(1, p_original)

def transform_probabilities(p, a=1.0, b=0.0):
    if a <= 0:
        raise ValueError('a must be positive to preserve ranking')
    p_safe = np.clip(np.asarray(p), 1e-12, 1 - 1e-12)
    return expit(a * logit(p_safe) + b)

def reliability_table(y_values, probabilities, n_bins=10):
    frame = pd.DataFrame({'y': y_values, 'p': probabilities})
    frame['bin'] = pd.cut(frame['p'], bins=np.linspace(0, 1, n_bins + 1),
                          include_lowest=True)
    return (frame.groupby('bin', observed=False)
            .agg(count=('y', 'size'), predicted=('p', 'mean'), observed=('y', 'mean'))
            .reset_index())

display(pd.DataFrame({'y': y, 'original_probability': p_original}).head())

## Investigation 1: Construct controlled alternative predictions

Build the three probability versions and confirm identical case order/rankings. Explain why strict monotonicity matters and verify the transformation on a few values.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 2: Assess ranking and probability quality

Create a compact table of AUC, log-loss and Brier score. Build reliability plots with counts and the diagonal reference. Omit empty-bin points and visually distinguish sparse bins using your stated convention. Distinguish finite-sample deviations from the deliberate transformations.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 3: Explain the consequence

Choose one practical use of numerical probabilities and show what the transformations would change. Explain why equal AUC does not imply equal probability quality or identical fixed-threshold decisions.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Your teaching conclusions

Replace this text with:

1. A direct answer to the tutorial's central question, supported by your results.
2. Two or three takeaways that classmates should remember.
3. One limitation, counterexample or condition under which the result changes.
4. One discussion question, plus notes on how your group would answer it.

## Contribution and verification record

- **Supplied material retained or changed:** …
- **Your group's experimental and explanatory contributions:** …
- **Independent check of an important result:** …
- **AI tools and their contributions, if any:** …
- **How generated code/explanations were checked:** …
- **Package versions, random seeds and reproduction instructions:** …
- **Group members and contributions:** …

## Before presenting

- [ ] Restarted the kernel and ran all cells successfully.
- [ ] Checked that splits, metrics, units and class labels are correct.
- [ ] Distinguished evidence from interpretation; avoided claims stronger than the experiment supports.
- [ ] Labelled plots and made figures legible for classmates.
- [ ] Rehearsed within 15 minutes and prepared for discussion.
- [ ] Included the completed notebook and any additional presentation material.